# Bloque 2: Transformaciones Complejas
## Módulo ETCDFM · Unidad Didáctica 3

**Extracción y carga desde bases de datos relacionales**

---

### Contenidos
1. Subqueries: escalar, fila y tabla
2. CTEs: `WITH` clause
3. Window functions: `OVER`, `PARTITION BY`, `LAG`/`LEAD`
4. `UNION`, `EXCEPT`, `INTERSECT`
5. pandas `merge` avanzado
6. `pivot_table` y `melt`
7. SQL + pandas: estrategias combinadas
8. Optimización: índices y `EXPLAIN QUERY PLAN`

> Como en el Bloque 1, todos los importes son **netos**: `cantidad × precio_unit × (1 − descuento_pct / 100)`.


## Subqueries (Subconsultas)

Una subquery es una consulta SQL anidada dentro de otra.

| Tipo | Posición | Devuelve | Ejemplo |
|---|---|---|---|
| **Escalar** | `SELECT`, `WHERE` | Un único valor | `WHERE precio > (SELECT AVG(precio) FROM productos)` |
| **Fila** | `WHERE` | Una fila | `WHERE (ciudad, segmento) = (SELECT ...)` |
| **Tabla derivada** | `FROM` (con alias) | Tabla virtual | `FROM (SELECT ...) AS sub` |
| **Correlacionada** | `WHERE` | Depende de la fila externa | `WHERE EXISTS (SELECT ... WHERE p.id_cliente = c.id_cliente)` |

```sql
-- Clientes que han gastado más que la media de los clientes
SELECT nombre, total_gastado
FROM (
    SELECT c.nombre,
           SUM(d.cantidad * d.precio_unit * (1 - p.descuento_pct / 100.0)) AS total_gastado
    FROM   clientes c
    JOIN   pedidos p         ON c.id_cliente = p.id_cliente
    JOIN   detalles_pedido d ON p.id_pedido  = d.id_pedido
    GROUP BY c.id_cliente
) AS resumen
WHERE total_gastado > (
    SELECT AVG(total) FROM (
        SELECT SUM(d.cantidad * d.precio_unit * (1 - p.descuento_pct / 100.0)) AS total
        FROM   pedidos p JOIN detalles_pedido d ON p.id_pedido = d.id_pedido
        GROUP BY p.id_cliente
    ) AS t      -- PostgreSQL (antes de la v16) exige alias en toda tabla derivada
);
```

> El alias `resumen` solo existe en el `FROM` donde se define: la subconsulta del `WHERE` no puede usarlo y hay que repetir el cálculo. Para no repetirlo usamos una CTE, en el apartado siguiente.

In [ ]:
import sqlite3
import pandas as pd

DB_TIENDA = '../Bloque_1_SQL_consultas_y_extraccion/datos/tienda.db'
DB_SENSORES = 'datos/sensores.db'

con = sqlite3.connect(DB_TIENDA)

# Tabla derivada en el FROM: productos con precio mayor que la media de su categoría
sql = """
    SELECT p.nombre, c.nombre AS categoria,
           ROUND(p.precio, 2) AS precio,
           ROUND(max_cat.maximo, 2) AS maximo_categoria
    FROM   productos p
    JOIN   categorias c ON p.id_categoria = c.id_categoria
    JOIN   (
        SELECT id_categoria, MAX(precio) AS maximo
        FROM   productos
        GROUP BY id_categoria
    ) AS max_cat ON p.id_categoria = max_cat.id_categoria
    WHERE  p.precio = max_cat.maximo
    ORDER  BY categoria, precio DESC
    LIMIT  12
"""
df = pd.read_sql(sql, con)
print('Productos por encima de la media de su categoría:')
print(df.to_string(index=False))

print()

# Subconsulta correlacionada: clientes con al menos 3 pedidos entregados
sql2 = """
    SELECT c.nombre, c.ciudad, c.segmento
    FROM   clientes c
    WHERE  (
        SELECT COUNT(*) FROM pedidos p
        WHERE  p.id_cliente = c.id_cliente
          AND  p.estado = 'entregado'
    ) >= 3
    ORDER  BY c.nombre
"""
df2 = pd.read_sql(sql2, con)
print(f'Clientes con 3 o más pedidos entregados: {len(df2)} (mostramos 10)')
print(df2.head(10).to_string(index=False))

print()

#clientes con al menos 1 pedidos entregados en la categoría de productos "Informatica"
sql4 = """
    SELECT c.nombre, c.ciudad, c.segmento
    FROM   clientes c
    WHERE  (
        SELECT COUNT(*)  FROM pedidos p
        JOIN   detalles_pedido dp ON p.id_pedido = dp.id_pedido
        JOIN   productos pr ON dp.id_producto = pr.id_producto
        WHERE  p.id_cliente = c.id_cliente
          AND  p.estado = 'entregado'
          AND  pr.nombre LIKE '%Informatica%'
    ) >= 1
    ORDER  BY c.nombre
"""
df4 = pd.read_sql(sql4, con)
print(f'Clientes con 3 o más pedidos entregados en la categoría de productos "Informatica": {len(df4)} (mostramos 10)')
print(df4.head(10).to_string(index=False))

    
# EXISTS / NOT EXISTS: número de clientes que nunca han cancelado un pedido
sql3 = """
    SELECT COUNT(*) AS n
    FROM   clientes c
    WHERE  EXISTS     (SELECT 1 FROM pedidos p WHERE p.id_cliente = c.id_cliente)
      AND  NOT EXISTS (SELECT 1 FROM pedidos p WHERE p.id_cliente = c.id_cliente
                                               AND p.estado = 'cancelado')
"""
print('Clientes con pedidos y ninguno cancelado:', pd.read_sql(sql3, con)['n'][0])

con.close()

Productos por encima de la media de su categoría:
        nombre    categoria  precio  maximo_categoria
     Te Matcha Alimentacion    18.5              18.5
 Bicicleta MTB     Deportes   649.0             649.0
  Smart TV 55"  Electronica   549.0             549.0
 Silla Oficina        Hogar   229.0             229.0
  Portatil Pro  Informatica  1199.0            1199.0
 Deep Learning       Libros    54.9              54.9
Smartphone X15      Moviles   799.0             799.0
Zapatillas Run         Ropa    89.9              89.9

Clientes con 3 o más pedidos entregados: 57 (mostramos 10)
          nombre     ciudad segmento
   Adriana Soler    Cordoba standard
       Aida Ruiz     Malaga standard
  Alberto Flores     Madrid standard
    Alicia Bravo      Palma  premium
  Alvaro Marquez   Zaragoza   basico
    Amparo Perez     Murcia standard
  Andres Jimenez    Cordoba   basico
Arturo Fernandez    Sevilla standard
  Beatriz Campos Las Palmas standard
    Carlos Lopez     Madrid standa

## CTEs: `WITH` Clause (Common Table Expressions)

Las CTEs dan nombre a subconsultas para **reutilizarlas** y mejorar la legibilidad.

```sql
WITH
  ventas_cliente AS (
    SELECT id_cliente, SUM(cantidad * precio_unit * (1 - descuento_pct / 100.0)) AS total
    FROM   pedidos JOIN detalles_pedido USING (id_pedido)
    WHERE  estado != 'cancelado'
    GROUP BY id_cliente
  ),
  deciles AS (
    SELECT *, NTILE(10) OVER (ORDER BY total DESC) AS decil
    FROM   ventas_cliente
  )
SELECT * FROM deciles WHERE decil = 1;  -- top 10%
```

**Ventajas sobre subqueries:**
- Se pueden referenciar **varias veces** en la misma consulta
- El optimizador puede materializarlas como tabla temporal
- Permiten **recursividad** (`WITH RECURSIVE`)

> 💡 En ML: las CTEs son perfectas para calcular **features agregadas** en SQL
> antes de cargar en pandas (análisis RFM, cohorts, medias móviles...).


In [2]:
con = sqlite3.connect(DB_TIENDA)

# CTE: análisis RFM (Recency, Frequency, Monetary) para segmentar clientes
# La recencia se mide en días desde la última compra hasta una fecha de referencia
FECHA_REF = '2025-01-01'   # primer día tras el periodo de datos (el último pedido es del 2024-12-30)

sql_rfm = """
WITH
  ultima_compra AS (
    SELECT id_cliente, MAX(fecha) AS ultima_fecha
    FROM   pedidos
    WHERE  estado != 'cancelado'
    GROUP BY id_cliente
  ),
  frecuencia AS (
    SELECT id_cliente, COUNT(DISTINCT id_pedido) AS n_pedidos
    FROM   pedidos
    WHERE  estado != 'cancelado'
    GROUP BY id_cliente
  ),
  monetario AS (
    SELECT p.id_cliente,
           ROUND(SUM(d.cantidad * d.precio_unit * (1 - p.descuento_pct / 100.0)), 2) AS total_gastado
    FROM   pedidos p
    JOIN   detalles_pedido d ON p.id_pedido = d.id_pedido
    WHERE  p.estado != 'cancelado'
    GROUP BY p.id_cliente
  )
SELECT
    c.nombre,
    c.segmento,
    CAST(julianday(:ref) - julianday(u.ultima_fecha) AS INTEGER) AS recencia_dias,
    f.n_pedidos                          AS frecuencia,
    m.total_gastado                      AS monetario
FROM   clientes c
JOIN   ultima_compra u ON c.id_cliente = u.id_cliente
JOIN   frecuencia    f ON c.id_cliente = f.id_cliente
JOIN   monetario     m ON c.id_cliente = m.id_cliente
ORDER  BY monetario DESC
LIMIT  15
"""
rfm = pd.read_sql(sql_rfm, con, params={'ref': FECHA_REF})
print('Análisis RFM (top 15 por gasto total):')
print(rfm.to_string(index=False))
con.close()

Análisis RFM (top 15 por gasto total):
         nombre segmento  recencia_dias  frecuencia  monetario
 Guillermo Cruz standard             30           8    9765.47
 Natalia Esteve   basico            270           4    9750.72
 Victor Fuentes  premium            183           5    9313.65
   Sonia Medina  premium             61           6    9295.55
      Aida Ruiz standard             44           9    8828.50
   Valeria Rios standard             18           6    8487.80
   Alicia Bravo  premium             60           5    8461.65
   Diego Torres standard             46           9    8409.58
 Tomas Carrasco  premium             30           6    7838.97
    Oscar Nunez standard             39           8    7609.60
   Manuel Ramos   basico             52           9    7128.14
    Juan Ortega standard            108           6    6841.84
  Adriana Soler standard            275           8    6501.53
Hector Iglesias standard            154           8    5949.73
   Roberto Diaz 

## Window Functions

Calculan valores **por fila** sin colapsar el resultado (a diferencia de `GROUP BY`).

```sql
funcion() OVER (
    [PARTITION BY columna]   -- 'grupos' independientes
    [ORDER BY columna]       -- orden dentro de cada particion
    [ROWS/RANGE ...]         -- marco de filas
)
```

| Función | Descripción |
|---|---|
| `ROW_NUMBER()` | Número de fila (1,2,3,...) |
| `RANK()` | Ranking con empates (1,1,3,...) |
| `DENSE_RANK()` | Ranking denso (1,1,2,...) |
| `NTILE(n)` | Divide en n grupos |
| `LAG(col, n)` | Valor n filas anterior |
| `LEAD(col, n)` | Valor n filas posterior |
| `SUM/AVG(col) OVER (...)` | Acumulado / media móvil |
| `FIRST_VALUE/LAST_VALUE(col)` | Primer / último valor de la ventana |


![Funciones de ventana frente a GROUP BY](imagenes/b2_window_vs_groupby.png)

In [3]:
con_s = sqlite3.connect(DB_SENSORES)

# Funciones de ventana sobre la serie horaria de un sensor de temperatura
# Los timestamps se guardan como texto ISO ('2024-01-01T00:00:00') y BETWEEN compara texto:
# '2024-01-01T23:00:00' <= '2024-01-02', pero '2024-01-02T00:00:00' > '2024-01-02',
# así que entran exactamente las 24 horas del día 1
sql_wf = """
    SELECT
        s.nombre                                              AS sensor,
        l.timestamp,
        ROUND(l.valor, 2)                                     AS valor,
        ROW_NUMBER() OVER (PARTITION BY l.id_sensor
                           ORDER BY l.timestamp)              AS fila,
        ROUND(LAG(l.valor) OVER (PARTITION BY l.id_sensor
                                 ORDER BY l.timestamp), 2)    AS valor_anterior,
        ROUND(l.valor - LAG(l.valor) OVER (PARTITION BY l.id_sensor
                                           ORDER BY l.timestamp), 3) AS variacion,
        ROUND(AVG(l.valor) OVER (PARTITION BY l.id_sensor
                                 ORDER BY l.timestamp
                                 ROWS BETWEEN 5 PRECEDING AND CURRENT ROW), 2) AS media_movil_6h
    FROM   lecturas l
    JOIN   sensores s ON l.id_sensor = s.id_sensor
    WHERE  l.id_sensor = 1
      AND  l.timestamp BETWEEN '2024-01-01' AND '2024-01-02'
      AND  l.calidad = 1
    ORDER  BY l.timestamp
"""
df_wf = pd.read_sql(sql_wf, con_s, parse_dates=['timestamp'])
print('Funciones de ventana sobre el sensor de temperatura (24 h):')
print(df_wf[['timestamp','valor','variacion','media_movil_6h']].to_string(index=False))
con_s.close()

Funciones de ventana sobre el sensor de temperatura (24 h):
          timestamp  valor  variacion  media_movil_6h
2024-01-01 00:00:00  17.83        NaN           17.83
2024-01-01 01:00:00  18.13       0.30           17.98
2024-01-01 02:00:00  18.22       0.09           18.06
2024-01-01 03:00:00  18.96       0.74           18.29
2024-01-01 04:00:00  20.16       1.20           18.66
2024-01-01 05:00:00  20.67       0.51           19.00
2024-01-01 06:00:00  22.37       1.70           19.75
2024-01-01 07:00:00  23.31       0.94           20.61
2024-01-01 08:00:00  23.61       0.30           21.51
2024-01-01 09:00:00  24.28       0.67           22.40
2024-01-01 10:00:00  25.49       1.21           23.29
2024-01-01 11:00:00  25.92       0.43           24.16
2024-01-01 12:00:00  25.36      -0.56           24.66
2024-01-01 13:00:00  25.75       0.39           25.07
2024-01-01 14:00:00  25.63      -0.12           25.41
2024-01-01 15:00:00  24.91      -0.72           25.51
2024-01-01 16:00:00  2

In [4]:
con = sqlite3.connect(DB_TIENDA)

# RANK de productos por facturación (revenue) dentro de cada categoría
sql_rank = """
WITH revenue AS (
    SELECT
        cat.nombre  AS categoria,
        pr.nombre   AS producto,
        ROUND(SUM(d.cantidad * d.precio_unit * (1 - p.descuento_pct / 100.0)), 2) AS revenue
    FROM   detalles_pedido d
    JOIN   pedidos p    ON d.id_pedido   = p.id_pedido
    JOIN   productos pr ON d.id_producto = pr.id_producto
    JOIN   categorias cat ON pr.id_categoria = cat.id_categoria
    WHERE  p.estado = 'entregado'
    GROUP BY cat.id_categoria, pr.id_producto
)
SELECT
    categoria, producto, revenue,
    RANK() OVER (PARTITION BY categoria ORDER BY revenue DESC) AS ranking_cat,
    ROUND(revenue * 100.0 / SUM(revenue) OVER (PARTITION BY categoria), 1) AS pct_cat
FROM revenue
ORDER BY categoria, ranking_cat
"""
df_rank = pd.read_sql(sql_rank, con)
print('Ranking de productos por facturación dentro de cada categoría (2 primeros):')
print(df_rank[df_rank['ranking_cat'] <= 2].to_string(index=False))
con.close()

Ranking de productos por facturación dentro de cada categoría (2 primeros):
   categoria          producto  revenue  ranking_cat  pct_cat
Alimentacion         Te Matcha   867.65            1     37.3
Alimentacion Cafe Especialidad   550.56            2     23.7
    Deportes     Bicicleta MTB 25278.55            1     81.3
    Deportes   Mancuernas 20kg  2017.48            2      6.5
 Electronica      Smart TV 55" 20093.40            1     46.7
 Electronica      Proyector HD 11284.70            2     26.3
       Hogar     Silla Oficina 10293.55            1     42.5
       Hogar   Mesa Escritorio  7002.45            2     28.9
 Informatica      Portatil Pro 54554.50            1     68.9
 Informatica       Monitor 27" 10323.00            2     13.0
      Libros  Data Engineering  2639.71            1     27.7
      Libros    Python para ML  1994.85            2     20.9
     Moviles    Smartphone X15 24968.75            1     59.1
     Moviles        Tablet 10"  8079.35            2    

## `UNION`, `EXCEPT`, `INTERSECT`

Operaciones de **conjuntos** sobre resultados de SELECT.

| Operador | Resultado | Duplicados |
|---|---|---|
| `UNION` | Filas de A **más** filas de B | Elimina duplicados |
| `UNION ALL` | Filas de A **más** filas de B | Conserva duplicados |
| `EXCEPT` (MINUS) | Filas en A que **no están** en B | Elimina duplicados |
| `INTERSECT` | Filas que están en A **y también** en B | Elimina duplicados |

```sql
-- Clientes premium O de Madrid (UNION)
SELECT id_cliente FROM clientes WHERE segmento = 'premium'
UNION
SELECT id_cliente FROM clientes WHERE ciudad = 'Madrid';

-- Clientes premium que NO son de Madrid (EXCEPT)
SELECT id_cliente FROM clientes WHERE segmento = 'premium'
EXCEPT
SELECT id_cliente FROM clientes WHERE ciudad = 'Madrid';
```

> Las columnas deben coincidir en número y tipo de dato entre los SELECT.


In [5]:
con = sqlite3.connect(DB_TIENDA)

# UNION: clientes que han comprado en 2023 O en 2024
sql_union = """
    SELECT id_cliente, '2023' AS ano
    FROM   pedidos
    WHERE  fecha LIKE '2023%' AND estado != 'cancelado'
    UNION
    SELECT id_cliente, '2024' AS ano
    FROM   pedidos
    WHERE  fecha LIKE '2024%' AND estado != 'cancelado'
    ORDER  BY id_cliente, ano
    LIMIT  12
"""
df_u = pd.read_sql(sql_union, con)
print('UNION — clientes con compras en 2023 o 2024 (muestra):')
print(df_u.to_string(index=False))

print()

# EXCEPT: clientes que compraron en 2023 pero NO en 2024
sql_exc = """
    SELECT DISTINCT id_cliente FROM pedidos
    WHERE  fecha LIKE '2023%' AND estado != 'cancelado'
    EXCEPT
    SELECT DISTINCT id_cliente FROM pedidos
    WHERE  fecha LIKE '2024%' AND estado != 'cancelado'
"""
df_e = pd.read_sql(sql_exc, con)
print(f'EXCEPT — clientes que solo compraron en 2023 (no en 2024): {len(df_e)}')

# INTERSECT: compraron en ambos años
sql_int = """
    SELECT DISTINCT id_cliente FROM pedidos
    WHERE  fecha LIKE '2023%' AND estado != 'cancelado'
    INTERSECT
    SELECT DISTINCT id_cliente FROM pedidos
    WHERE  fecha LIKE '2024%' AND estado != 'cancelado'
"""
df_i = pd.read_sql(sql_int, con)
print(f'INTERSECT — clientes que compraron en 2023 Y 2024: {len(df_i)}')
con.close()

UNION — clientes con compras en 2023 o 2024 (muestra):
 id_cliente  ano
          1 2024
          2 2023
          2 2024
          3 2023
          3 2024
          4 2024
          5 2023
          5 2024
          6 2023
          7 2023
          7 2024
          8 2023

EXCEPT — clientes que solo compraron en 2023 (no en 2024): 9
INTERSECT — clientes que compraron en 2023 Y 2024: 83


## pandas `merge`: Combinación de DataFrames

Equivalente Python a los JOINs SQL:

```python
pd.merge(
    left_df, right_df,
    on='clave',            # columna/s de unión
    how='inner',           # 'inner' | 'left' | 'right' | 'outer'
    suffixes=('_x','_y')   # sufijos si hay columnas duplicadas
)
```

| `how=` | SQL equivalente | Filas resultado |
|---|---|---|
| `'inner'` | `INNER JOIN` | Intersección |
| `'left'` | `LEFT JOIN` | Todas las de izq. + matches |
| `'right'` | `RIGHT JOIN` | Todas las de der. + matches |
| `'outer'` | `FULL OUTER JOIN` | Unión completa |
| `'cross'` | `CROSS JOIN` | Producto cartesiano |

**Alternativas:**
- `df.join(other, on='key')` — join por índice
- `pd.concat([df1, df2])` — apilado vertical (equivale a `UNION ALL`)


In [6]:
con = sqlite3.connect(DB_TIENDA)

# Cargar tablas necesarias
clientes = pd.read_sql('SELECT * FROM clientes', con)
pedidos  = pd.read_sql('SELECT * FROM pedidos', con)
detalles = pd.read_sql('SELECT * FROM detalles_pedido', con)
productos = pd.read_sql('SELECT * FROM productos', con)
con.close()

print(f'Tablas cargadas: clientes={len(clientes)}, pedidos={len(pedidos)},',
      f'detalles={len(detalles)}, productos={len(productos)}')

# INNER MERGE: solo pedidos con cliente conocido
ped_cli = pd.merge(pedidos, clientes[['id_cliente','nombre','segmento','ciudad']],
                   on='id_cliente', how='inner')
print(f'\nINNER merge pedidos + clientes: {len(ped_cli)} filas')

# LEFT MERGE: todos los clientes, incluso sin pedidos
cli_ped = pd.merge(clientes[['id_cliente','nombre','segmento']],
                   pedidos.groupby('id_cliente').size().reset_index(name='n_pedidos'),
                   on='id_cliente', how='left')
cli_ped['n_pedidos'] = cli_ped['n_pedidos'].fillna(0).astype(int)
print(f'LEFT merge clientes + conteo pedidos: {len(cli_ped)} filas')
print(f'  Clientes sin pedidos: {(cli_ped.n_pedidos == 0).sum()}')

# Cadena de merges: ticket completo
ticket = (
    detalles
    .merge(pedidos[['id_pedido','id_cliente','fecha','estado','descuento_pct']], on='id_pedido')
    .merge(clientes[['id_cliente','nombre','segmento']], on='id_cliente')
    .merge(productos[['id_producto','nombre','id_categoria']].rename(
           columns={'nombre':'producto'}), on='id_producto')
)
ticket['importe'] = ticket['cantidad'] * ticket['precio_unit'] * (1 - ticket['descuento_pct'] / 100)
print(f'\nTicket completo (merge chain): {len(ticket)} filas, columnas: {list(ticket.columns)}')


Tablas cargadas: clientes=100, pedidos=500, detalles=1261, productos=41

INNER merge pedidos + clientes: 500 filas
LEFT merge clientes + conteo pedidos: 100 filas
  Clientes sin pedidos: 2

Ticket completo (merge chain): 1261 filas, columnas: ['id_detalle', 'id_pedido', 'id_producto', 'cantidad', 'precio_unit', 'id_cliente', 'fecha', 'estado', 'descuento_pct', 'nombre', 'segmento', 'producto', 'id_categoria', 'importe']


## `pivot_table` y `melt`

### `pivot_table`: filas → columnas (wide format)
```python
df.pivot_table(
    values='importe',
    index='mes',           # filas
    columns='segmento',    # columnas
    aggfunc='sum',         # sum | mean | count | ...
    fill_value=0
)
```

### `melt`: columnas → filas (long/tidy format)
```python
df.melt(
    id_vars=['mes'],        # columnas que permanecen
    value_vars=['A','B'],   # columnas que se 'derriten'
    var_name='segmento',
    value_name='importe'
)
```

> Para entrenar, scikit-learn espera una **matriz wide**: una fila por muestra y una columna por variable. Pasar de long a wide con `pivot_table` es la forma de convertir registros de eventos en variables. El formato **long** (tidy) es el cómodo para agrupar, filtrar y dibujar con seaborn, y el que usan las fórmulas de statsmodels.

In [7]:
# Extraemos con SQL las líneas entregadas con su fecha, segmento y categoría
con = sqlite3.connect(DB_TIENDA)
ticket = pd.read_sql("""
    SELECT p.fecha, c.segmento, cat.nombre AS categoria,
           ROUND(d.cantidad * d.precio_unit * (1 - p.descuento_pct / 100.0), 2) AS importe
    FROM   detalles_pedido d
    JOIN   pedidos p     ON d.id_pedido   = p.id_pedido
    JOIN   clientes c    ON p.id_cliente  = c.id_cliente
    JOIN   productos pr  ON d.id_producto = pr.id_producto
    JOIN   categorias cat ON pr.id_categoria = cat.id_categoria
    WHERE  p.estado = 'entregado'
""", con, parse_dates=['fecha'])
con.close()
ticket['mes'] = ticket['fecha'].dt.to_period('M').astype(str)

# pivot_table: facturación mensual por segmento
pivot = ticket.pivot_table(
    values='importe', index='mes', columns='segmento',
    aggfunc='sum', fill_value=0
).round(0).astype(int)
print('Facturación mensual por segmento (EUR):')
print(pivot.head(8).to_string())

print()

# melt: volver a formato long
long_df = pivot.reset_index().melt(
    id_vars='mes', var_name='segmento', value_name='importe'
)
print('Formato long (tidy):')
print(long_df.head(9).to_string(index=False))
print(f'Shape: {long_df.shape}')

Facturación mensual por segmento (EUR):
segmento  basico  premium  standard
mes                                
2023-01     1088     2896      2843
2023-02     1143      175      4226
2023-03     7267     8540      2370
2023-04     4048      208      4476
2023-05     1902     1157      2482
2023-06     3052     2410      1299
2023-07     9219      232      4271
2023-08     1070     1805      3077

Formato long (tidy):
    mes segmento  importe
2023-01   basico     1088
2023-02   basico     1143
2023-03   basico     7267
2023-04   basico     4048
2023-05   basico     1902
2023-06   basico     3052
2023-07   basico     9219
2023-08   basico     1070
2023-09   basico     4035
Shape: (72, 3)


## SQL + pandas: Estrategias Combinadas

**Regla práctica: filtrar y agregar en SQL, transformar y modelar en pandas.**

| Tarea | SQL | pandas |
|---|---|---|
| Selección de filas | `WHERE`, `HAVING` | `.loc`, `.query` |
| Proyección de columnas | `SELECT col1, col2` | `[['col1','col2']]` |
| Agregación simple | `GROUP BY` + `SUM/COUNT` | `.groupby().agg()` |
| Window functions | `OVER (PARTITION BY ...)` | `.groupby().transform()` |
| Normalización / one-hot | ✗ complejo | `StandardScaler`, `pd.get_dummies` |
| Imputación de nulos | ✗ limitado | `.fillna`, `SimpleImputer` |
| Feature engineering ML | ✗ complejo | libre |

**Patron "Extract-Then-Transform":**
```
SQL (en BD) → filtrar → agregar → pandas → feature engineering → ML
```

> Con millones de filas: ejecutar el máximo posible en SQL
> antes de traer datos a Python.


In [8]:
# Paso 1: extraer con SQL (solo lo necesario)
con = sqlite3.connect(DB_TIENDA)
features_sql = pd.read_sql("""
    WITH base AS (
        SELECT
            c.id_cliente,
            c.segmento,
            c.ciudad,
            COUNT(DISTINCT p.id_pedido)               AS n_pedidos,
            ROUND(SUM(d.cantidad * d.precio_unit * (1 - p.descuento_pct / 100.0)), 2) AS gasto_total,
            ROUND(AVG(d.precio_unit), 2)               AS precio_medio,   -- precio medio por línea, sin ponderar por cantidad
            MAX(p.fecha)                               AS ultima_compra
        FROM clientes c
        JOIN pedidos p           ON c.id_cliente  = p.id_cliente
        JOIN detalles_pedido d   ON p.id_pedido   = d.id_pedido
        WHERE p.estado != 'cancelado'
        GROUP BY c.id_cliente
    )
    SELECT * FROM base
""", con, parse_dates=['ultima_compra'])
con.close()

print(f'Variables extraídas con SQL: {features_sql.shape}  (98 clientes: los 2 sin pedidos quedan fuera por el JOIN)')
print(features_sql.head(5).to_string(index=False))

# Paso 2: transformar en pandas
today = pd.Timestamp('2025-01-01')   # fecha de referencia, como en el RFM
features_sql['dias_desde_compra'] = (today - features_sql['ultima_compra']).dt.days
features_sql['ticket_medio'] = (features_sql['gasto_total'] / features_sql['n_pedidos']).round(2)
features_sql = pd.get_dummies(features_sql, columns=['segmento', 'ciudad'], drop_first=True)

print(f'\nVariables tras la transformación en pandas: {features_sql.shape}')
print('Columnas:', list(features_sql.columns))

Variables extraídas con SQL: (98, 7)  (98 clientes: los 2 sin pedidos quedan fuera por el JOIN)
 id_cliente segmento    ciudad  n_pedidos  gasto_total  precio_medio ultima_compra
          1 standard    Malaga          2      1314.65        103.44    2024-11-04
          2 standard    Madrid          8      2670.09         66.28    2024-11-21
          3  premium Barcelona          4      1357.25         80.59    2024-11-19
          4   basico  Zaragoza          2       403.55         69.67    2024-05-24
          5 standard   Sevilla          8      5453.23        155.99    2024-08-09

Variables tras la transformación en pandas: (98, 20)
Columnas: ['id_cliente', 'n_pedidos', 'gasto_total', 'precio_medio', 'ultima_compra', 'dias_desde_compra', 'ticket_medio', 'segmento_premium', 'segmento_standard', 'ciudad_Barcelona', 'ciudad_Bilbao', 'ciudad_Cordoba', 'ciudad_Las Palmas', 'ciudad_Madrid', 'ciudad_Malaga', 'ciudad_Murcia', 'ciudad_Palma', 'ciudad_Sevilla', 'ciudad_Valencia', 'ciudad_

## Optimización: Índices y `EXPLAIN QUERY PLAN`

### ¿Por qué son lentas algunas consultas?
- Sin índice → **table scan**: la BD lee **todas** las filas
- Con índice → **index seek**: acceso directo en O(log n)

```sql
-- Ver el plan de ejecución
EXPLAIN QUERY PLAN
SELECT * FROM pedidos WHERE fecha = '2024-06-01';
-- Sin índice: SCAN pedidos
-- Con índice: SEARCH pedidos USING INDEX idx_pedidos_fecha (fecha=?)
```

**Cuándo crear índices:**
- Columnas usadas frecuentemente en `WHERE`, `JOIN ON`, `ORDER BY`
- Columnas de alta cardinalidad (muchos valores distintos)
- **NO** en columnas con muy pocos valores distintos (boolean, estado con 4 valores)
- **NO** en tablas pequeñas (overhead > beneficio)

```sql
CREATE INDEX idx_nombre ON tabla(columna);
CREATE UNIQUE INDEX idx_email ON clientes(email);
CREATE INDEX idx_compuesto ON pedidos(id_cliente, fecha);
DROP INDEX idx_nombre;
```

In [9]:
import time, shutil, tempfile, os

# Trabajamos sobre una COPIA de sensores.db: crear índices modifica el fichero
# y no queremos alterar el que viene con el repositorio
DB_COPIA = os.path.join(tempfile.gettempdir(), 'sensores_indices.db')
shutil.copy(DB_SENSORES, DB_COPIA)
con_s = sqlite3.connect(DB_COPIA)
cur = con_s.cursor()

sql_bench = """
    SELECT id_sensor, COUNT(*), ROUND(AVG(valor), 3)
    FROM   lecturas
    WHERE  timestamp BETWEEN '2024-01-15' AND '2024-02-15'
      AND  calidad = 1
    GROUP BY id_sensor
"""

def plan_y_tiempo(etiqueta):
    print(f'=== {etiqueta} ===')
    for row in cur.execute('EXPLAIN QUERY PLAN ' + sql_bench):
        print('  ', row[-1])
    tiempos = []
    for _ in range(5):
        t0 = time.perf_counter()
        pd.read_sql(sql_bench, con_s)
        tiempos.append((time.perf_counter() - t0) * 1000)
    print(f'   {sum(tiempos)/len(tiempos):.2f} ms (media de 5 ejecuciones)\n')

# 1. Sin ningún índice sobre timestamp: recorrido completo de la tabla
cur.execute('DROP INDEX IF EXISTS idx_lecturas_ts')
plan_y_tiempo('Sin índice en timestamp')

# 2. Índice simple en timestamp
cur.execute('CREATE INDEX idx_lecturas_ts ON lecturas(timestamp)')
plan_y_tiempo('Con idx_lecturas_ts (timestamp)')

# 3. Índice compuesto (timestamp, calidad)
cur.execute('CREATE INDEX idx_lecturas_ts_calidad ON lecturas(timestamp, calidad)')
plan_y_tiempo('Con idx_lecturas_ts_calidad (timestamp, calidad)')

print('El índice compuesto apenas ayuda aquí: casi todas las lecturas tienen calidad = 1,')
print('así que filtrar por calidad no descarta filas. Un índice compensa cuando el filtro es selectivo.')

con_s.close()
os.remove(DB_COPIA)

=== Sin índice en timestamp ===
   SCAN lecturas USING INDEX idx_lecturas_sensor
   2.01 ms (media de 5 ejecuciones)

=== Con idx_lecturas_ts (timestamp) ===
   SEARCH lecturas USING INDEX idx_lecturas_ts (timestamp>? AND timestamp<?)
   USE TEMP B-TREE FOR GROUP BY
   1.28 ms (media de 5 ejecuciones)

=== Con idx_lecturas_ts_calidad (timestamp, calidad) ===
   SEARCH lecturas USING INDEX idx_lecturas_ts_calidad (timestamp>? AND timestamp<?)
   USE TEMP B-TREE FOR GROUP BY
   1.13 ms (media de 5 ejecuciones)

El índice compuesto apenas ayuda aquí: casi todas las lecturas tienen calidad = 1,
así que filtrar por calidad no descarta filas. Un índice compensa cuando el filtro es selectivo.


## Resumen del Bloque 2

| Técnica | SQL | pandas |
|---|---|---|
| Subconsultas | Subquery en `WHERE`/`FROM` | `.query()` + lambda |
| Tablas temporales | `WITH cte AS (...)` | variable intermedia |
| Window functions | `OVER (PARTITION BY ...)` | `.groupby().transform()` |
| Conjuntos | `UNION`, `EXCEPT`, `INTERSECT` | `pd.concat`, `isin`, `~isin` |
| Combinación | `JOIN` | `pd.merge(how=...)` |
| Reestructurar | `CASE WHEN` + `GROUP BY` | `pivot_table` / `melt` |
| Optimizar | `CREATE INDEX`, `EXPLAIN` | categoricals, dtypes |

---

**Próximo bloque →** Carga de datos y feature stores:
DDL, DML, df.to_sql, carga masiva, feature stores, pipeline ETL completo.
